In [169]:
import pandas as pd
import numpy as np

# 1. Configuración Inicial y Semilla
np.random.seed(42)
n_appointments = 800000
n_patients = 25000
current_year = 2026

print("🇲🇽 Generando Data Warehouse Hospitalario (Versión Final Consistente)... 🏥")

# --- 2. DICCIONARIO MAESTRO DE DIAGNÓSTICOS (Dimensión Diagnoses) ---
diag_config = {
    'Acute Myocardial Infarction': {'id': 'ICD-101', 'dept': 'Cardiology', 'contagious': 0, 'risk': 9, 'min_d': 5, 'max_d': 12, 'cost': 85000, 'urgency': 'Red', 'recovery': 12},
    'Hypertensive Crisis':         {'id': 'ICD-102', 'dept': 'Cardiology', 'contagious': 0, 'risk': 7, 'min_d': 1, 'max_d': 3,  'cost': 15000, 'urgency': 'Orange', 'recovery': 2},
    'Bacterial Pneumonia':        {'id': 'ICD-103', 'dept': 'Respiratory', 'contagious': 1, 'risk': 6, 'min_d': 3, 'max_d': 7,  'cost': 35000, 'urgency': 'Orange', 'recovery': 4},
    'Acute Bronchitis':           {'id': 'ICD-104', 'dept': 'Respiratory', 'contagious': 1, 'risk': 3, 'min_d': 0.5, 'max_d': 2, 'cost': 5000,  'urgency': 'Yellow', 'recovery': 2},
    'Pulmonary Tuberculosis':     {'id': 'ICD-105', 'dept': 'Respiratory', 'contagious': 1, 'risk': 8, 'min_d': 7, 'max_d': 21, 'cost': 60000, 'urgency': 'Orange', 'recovery': 24},
    'Femur Fracture':             {'id': 'ICD-106', 'dept': 'Traumatology', 'contagious': 0, 'risk': 5, 'min_d': 2, 'max_d': 5,  'cost': 45000, 'urgency': 'Orange', 'recovery': 8},
    'Hip Fracture':               {'id': 'ICD-107', 'dept': 'Traumatology', 'contagious': 0, 'risk': 7, 'min_d': 4, 'max_d': 10, 'cost': 75000, 'urgency': 'Orange', 'recovery': 16},
    'Ankle Sprain':               {'id': 'ICD-108', 'dept': 'Traumatology', 'contagious': 0, 'risk': 2, 'min_d': 0.1, 'max_d': 0.5, 'cost': 3000, 'urgency': 'Green', 'recovery': 3},
    'Acute Appendicitis':         {'id': 'ICD-109', 'dept': 'Surgery', 'contagious': 0, 'risk': 8, 'min_d': 2, 'max_d': 4,  'cost': 28000, 'urgency': 'Orange', 'recovery': 4},
    'Acute Gastroenteritis':      {'id': 'ICD-110', 'dept': 'Internal Medicine', 'contagious': 1, 'risk': 3, 'min_d': 0.2, 'max_d': 1, 'cost': 4000, 'urgency': 'Yellow', 'recovery': 1},
    'Cholecystitis':              {'id': 'ICD-111', 'dept': 'Surgery', 'contagious': 0, 'risk': 6, 'min_d': 2, 'max_d': 4,  'cost': 32000, 'urgency': 'Orange', 'recovery': 4},
    'Type 2 Diabetes Mellitus':   {'id': 'ICD-112', 'dept': 'Endocrinology', 'contagious': 0, 'risk': 7, 'min_d': 1, 'max_d': 3,  'cost': 12000, 'urgency': 'Yellow', 'recovery': 52},
    'Urinary Tract Infection':    {'id': 'ICD-113', 'dept': 'Internal Medicine', 'contagious': 0, 'risk': 3, 'min_d': 0.2, 'max_d': 1, 'cost': 3500, 'urgency': 'Yellow', 'recovery': 1},
    'Common Flu':                 {'id': 'ICD-114', 'dept': 'Internal Medicine', 'contagious': 1, 'risk': 2, 'min_d': 0.1, 'max_d': 0.5, 'cost': 800, 'urgency': 'Green', 'recovery': 1},
    'General Check-up':           {'id': 'ICD-115', 'dept': 'Preventive Medicine', 'contagious': 0, 'risk': 1, 'min_d': 0, 'max_d': 0.1, 'cost': 1500, 'urgency': 'Blue', 'recovery': 0}
}

# --- 3. GENERACIÓN DE PATIENTS_DIM (Con Lógica de Edad y Ocupación) ---
official_patient_ids = [f"PAT-25-{i:05d}" for i in range(1, n_patients + 1)]
birth_dates = pd.to_datetime('1945-01-01') + pd.to_timedelta(np.random.randint(0, 29000, n_patients), unit='d')
ages = [current_year - d.year for d in birth_dates]

# Reglas de Ocupación para evitar inconsistencias
occupations = []
for age in ages:
    if age < 6: occupations.append('Infant')
    elif age < 23: occupations.append('Student')
    elif age < 65: occupations.append(np.random.choice(['Employee', 'Self-employed', 'Unemployed'], p=[0.7, 0.2, 0.1]))
    else: occupations.append('Retired')

patients_df = pd.DataFrame({
    'patient_id': official_patient_ids,
    'full_name': [f'Patient_{i}' for i in range(1, n_patients + 1)],
    'gender': np.random.choice(['M', 'F', 'NB', ' '], n_patients, p=[0.48, 0.48, 0.02, 0.02]),
    'birth_date': birth_dates,
    'occupation': occupations,
    'state_mx': np.random.choice(['CDMX', 'Jalisco', 'Nuevo León', 'Puebla', 'Veracruz', 'Yucatán', 'Querétaro', 'Chihuahua'], n_patients),
    'blood_type': np.random.choice(['O+', 'A+', 'B+', 'O-', 'AB+'], n_patients),
    'height_m': np.random.normal(1.68, 0.08, n_patients).round(2),
    'weight_kg': np.random.normal(75, 15, n_patients).round(1),
    'insurance_provider': np.random.choice(['IMSS', 'ISSSTE', 'AXA', 'GNP', 'Out-of-pocket'], n_patients)
})

# --- 4. GENERACIÓN DE DIAGNOSES_DIM ---
diagnosis_data = [
    {
        'icd10_id': conf['id'],
        'diagnosis_name': name,
        'medical_specialty': conf['dept'],
        'risk_score': conf['risk'],
        'is_contagious': conf['contagious'],
        'avg_recovery_weeks': conf['recovery']
    }
    for name, conf in diag_config.items()
]
diagnoses_df = pd.DataFrame(diagnosis_data)

# --- 5. GENERACIÓN DE EMERGENCY_APPOINTMENTS (Tabla de Hechos) ---
# Sincronización absoluta: Solo usamos IDs existentes de las dimensiones
chosen_diagnoses_names = np.random.choice(list(diag_config.keys()), n_appointments)
chosen_patient_ids = np.random.choice(official_patient_ids, n_appointments)

icd_ids_fact = [diag_config[name]['id'] for name in chosen_diagnoses_names]
triage_fact = [diag_config[name]['urgency'] for name in chosen_diagnoses_names]
base_costs = [diag_config[name]['cost'] for name in chosen_diagnoses_names]

appointments_df = pd.DataFrame({
    'appointment_id': range(1, n_appointments + 1),
    'patient_id': chosen_patient_ids,
    'icd10_id': icd_ids_fact,
    'admission_date': pd.to_datetime('2025-01-01') + pd.to_timedelta(np.random.randint(0, 525600, n_appointments), unit='m'),
    'triage_level': triage_fact,
    'systolic_bp': np.random.randint(90, 180, n_appointments),
    'diastolic_bp': np.random.randint(60, 110, n_appointments),
    'heart_rate': np.random.randint(50, 120, n_appointments),
    'temp_celsius': np.random.normal(36.6, 1.2, n_appointments).round(1),
    'total_cost': (np.array(base_costs) * np.random.uniform(0.9, 1.2, n_appointments)).round(2),
    'hospital_branch': np.random.choice(['Centro', 'Santa Fe', 'Polanco', 'Monterrey', 'Guadalajara'], n_appointments)
})

# 6. Inyectar Ruido para Limpieza de Datos (Opcional)
appointments_df.loc[appointments_df.sample(frac=0.01).index, 'icd10_id'] = np.nan

# 7. Guardado
appointments_df.to_csv('emergency_appointments.csv', index=False)
patients_df.to_csv('patients_dim.csv', index=False)
diagnoses_df.to_csv('diagnoses_dim.csv', index=False)

print("\n✅ Proceso Finalizado. Archivos generados con éxito.")
print(f"Total Pacientes: {len(patients_df):,}")
print(f"Total Citas: {len(appointments_df):,}")

🇲🇽 Generando Data Warehouse Hospitalario (Versión Final Consistente)... 🏥

✅ Proceso Finalizado. Archivos generados con éxito.
Total Pacientes: 25,000
Total Citas: 800,000


In [170]:
unique_names = diagnoses_df['diagnosis_name'].nunique()
unique_ids = diagnoses_df['icd10_id'].nunique()

print(f"Nombres únicos: {unique_names}")
print(f"IDs únicos: {unique_ids}")

if unique_names == unique_ids:
    print("✅ Verificación exitosa: Cada diagnóstico tiene un ID único.")
else:
    print("❌ Error: Hay una inconsistencia entre nombres e IDs.")

Nombres únicos: 15
IDs únicos: 15
✅ Verificación exitosa: Cada diagnóstico tiene un ID único.


In [171]:
diagnoses_df.head(10)

,icd10_id,diagnosis_name,medical_specialty,risk_score,is_contagious,avg_recovery_weeks
0,ICD-101,Acute Myocardial Infarction,Cardiology,9,0,12
1,ICD-102,Hypertensive Crisis,Cardiology,7,0,2
2,ICD-103,Bacterial Pneumonia,Respiratory,6,1,4
3,ICD-104,Acute Bronchitis,Respiratory,3,1,2
4,ICD-105,Pulmonary Tuberculosis,Respiratory,8,1,24
5,ICD-106,Femur Fracture,Traumatology,5,0,8
6,ICD-107,Hip Fracture,Traumatology,7,0,16
7,ICD-108,Ankle Sprain,Traumatology,2,0,3
8,ICD-109,Acute Appendicitis,Surgery,8,0,4
9,ICD-110,Acute Gastroenteritis,Internal Medicine,3,1,1


In [172]:
# Verificación rápida de la dimensión
print(diagnoses_df[['icd10_id', 'diagnosis_name', 'medical_specialty']].sort_values('icd10_id').to_string(index=False))

icd10_id              diagnosis_name   medical_specialty
 ICD-101 Acute Myocardial Infarction          Cardiology
 ICD-102         Hypertensive Crisis          Cardiology
 ICD-103         Bacterial Pneumonia         Respiratory
 ICD-104            Acute Bronchitis         Respiratory
 ICD-105      Pulmonary Tuberculosis         Respiratory
 ICD-106              Femur Fracture        Traumatology
 ICD-107                Hip Fracture        Traumatology
 ICD-108                Ankle Sprain        Traumatology
 ICD-109          Acute Appendicitis             Surgery
 ICD-110       Acute Gastroenteritis   Internal Medicine
 ICD-111               Cholecystitis             Surgery
 ICD-112    Type 2 Diabetes Mellitus       Endocrinology
 ICD-113     Urinary Tract Infection   Internal Medicine
 ICD-114                  Common Flu   Internal Medicine
 ICD-115            General Check-up Preventive Medicine


In [173]:
appointments_df[appointments_df["patient_id"] == "PAT-25-00001"]

,appointment_id,patient_id,icd10_id,admission_date,triage_level,systolic_bp,diastolic_bp,heart_rate,temp_celsius,total_cost,hospital_branch
15039,15040,PAT-25-00001,ICD-101,2025-11-15 04:23:00,Red,122,65,116,35.2,97883.63,Guadalajara
26981,26982,PAT-25-00001,ICD-106,2025-11-11 11:13:00,Orange,127,103,87,36.8,44823.30,Monterrey
40343,40344,PAT-25-00001,ICD-103,2025-08-20 05:58:00,Orange,139,70,51,35.5,40477.51,Polanco
86184,86185,PAT-25-00001,ICD-107,2025-08-13 18:42:00,Orange,99,85,105,37.6,89190.41,Monterrey
108695,108696,PAT-25-00001,ICD-112,2025-05-20 17:56:00,Yellow,128,77,69,36.9,13044.56,Guadalajara
113683,113684,PAT-25-00001,ICD-104,2025-02-01 00:19:00,Yellow,114,104,54,38.8,5394.17,Polanco
131265,131266,PAT-25-00001,ICD-112,2025-12-07 06:01:00,Yellow,100,60,81,36.4,12209.39,Monterrey
156564,156565,PAT-25-00001,ICD-115,2025-08-07 15:29:00,Blue,93,69,100,36.1,1396.87,Monterrey
170362,170363,PAT-25-00001,ICD-106,2025-04-01 01:05:00,Orange,120,102,89,37.8,52044.15,Monterrey
174270,174271,PAT-25-00001,ICD-110,2025-03-09 21:54:00,Yellow,148,89,70,36.5,3805.18,Polanco


In [174]:
patients_df.head(10)

,patient_id,full_name,gender,birth_date,occupation,state_mx,blood_type,height_m,weight_kg,insurance_provider
0,PAT-25-00001,Patient_1,F,2009-10-06,Student,CDMX,B+,1.63,91.7,ISSSTE
1,PAT-25-00002,Patient_2,F,1988-03-31,Self-employed,Puebla,A+,1.63,64.2,GNP
2,PAT-25-00003,Patient_3,F,1947-05-11,Retired,Puebla,A+,1.72,74.7,IMSS
3,PAT-25-00004,Patient_4,M,1959-10-05,Retired,Jalisco,AB+,1.62,57.5,AXA
4,PAT-25-00005,Patient_5,F,2004-01-27,Student,Veracruz,O+,1.65,69.9,AXA
5,PAT-25-00006,Patient_6,M,1977-10-04,Employee,Jalisco,B+,1.56,52.4,AXA
6,PAT-25-00007,Patient_7,NB,1975-11-24,Self-employed,CDMX,O-,1.57,102.7,GNP
7,PAT-25-00008,Patient_8,M,2005-07-23,Student,Jalisco,B+,1.75,53.3,IMSS
8,PAT-25-00009,Patient_9,M,1962-02-26,Employee,Puebla,AB+,1.67,71.5,IMSS
9,PAT-25-00010,Patient_10,M,1991-02-19,Employee,Querétaro,O-,1.60,44.3,GNP


In [175]:
patients_df[patients_df["patient_id"] == "PAT-25-00001"]

,patient_id,full_name,gender,birth_date,occupation,state_mx,blood_type,height_m,weight_kg,insurance_provider
0,PAT-25-00001,Patient_1,F,2009-10-06,Student,CDMX,B+,1.63,91.7,ISSSTE


In [176]:
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   patient_id          25000 non-null  object        
 1   full_name           25000 non-null  object        
 2   gender              25000 non-null  object        
 3   birth_date          25000 non-null  datetime64[ns]
 4   occupation          25000 non-null  object        
 5   state_mx            25000 non-null  object        
 6   blood_type          25000 non-null  object        
 7   height_m            25000 non-null  float64       
 8   weight_kg           25000 non-null  float64       
 9   insurance_provider  25000 non-null  object        
dtypes: datetime64[ns](1), float64(2), object(7)
memory usage: 1.9+ MB


In [177]:
patients_df[patients_df["occupation"] == "Retired"][["birth_date","occupation"]]

,birth_date,occupation
2,1947-05-11,Retired
3,1959-10-05,Retired
10,1957-02-13,Retired
18,1949-08-13,Retired
19,1947-02-09,Retired
...,...,...
24984,1954-10-17,Retired
24987,1960-09-07,Retired
24988,1945-02-02,Retired
24991,1956-12-22,Retired


In [178]:
import pandas as pd

# 1. Carga de datos
df_appointments = pd.read_csv('emergency_appointments.csv')
df_patients = pd.read_csv('patients_dim.csv')
df_diagnoses = pd.read_csv('diagnoses_dim.csv')

def run_audit():
    print("=== AUDITORÍA DE SISTEMA HOSPITALARIO ===\n")

    # --- PRUEBA 1: Integridad Referencial (Foreign Keys) ---
    # ¿Existen IDs en Appointments que NO estén en las dimensiones?
    invalid_patients = df_appointments[~df_appointments['patient_id'].isin(df_patients['patient_id'])]['patient_id'].dropna().unique()
    # Filtramos nulos en diagnósticos porque los inyectamos a propósito para limpieza
    invalid_diagnoses = df_appointments[~df_appointments['icd10_id'].isin(df_diagnoses['icd10_id'])]['icd10_id'].dropna().unique()

    print(f"1. ÍNTEGRIDAD REFERENCIAL:")
    print(f"   - IDs de pacientes huérfanos: {len(invalid_patients)}")
    print(f"   - IDs de diagnósticos huérfanos: {len(invalid_diagnoses)}")
    if len(invalid_patients) == 0 and len(invalid_diagnoses) == 0:
        print("   ✅ Integridad referencial perfecta.\n")

    # --- PRUEBA 2: Validación de Relaciones (Cardinalidad) ---
    print(f"2. ANÁLISIS DE CARDINALIDAD:")

    # Relación 1:N entre Pacientes y Citas
    citas_por_paciente = df_appointments.groupby('patient_id').size()
    print(f"   - Promedio de citas por paciente: {citas_por_paciente.mean():.2f}")
    print(f"   - Paciente con más recurrencia: {citas_por_paciente.max()} citas")

    if df_patients['patient_id'].is_unique:
        print("   ✅ Dimensión Pacientes: patient_id es ÚNICO (Relación 1:N confirmada).")

    # Relación 1:N entre Diagnósticos y Citas
    if df_diagnoses['icd10_id'].is_unique:
        print("   ✅ Dimensión Diagnoses: icd10_id es ÚNICO (Relación 1:N confirmada).\n")

    # --- PRUEBA 3: Coherencia Lógica (Edad vs Ocupación) ---
    # Cruzamos con pacientes para validar la regla de negocio que corregimos
    check_logic = pd.merge(df_appointments[['patient_id']], df_patients[['patient_id', 'birth_date', 'occupation']], on='patient_id')
    check_logic['age'] = 2026 - pd.to_datetime(check_logic['birth_date']).dt.year

    inconsistencias = check_logic[(check_logic['occupation'] == 'Retired') & (check_logic['age'] < 60)]

    print(f"3. COHERENCIA DE NEGOCIO:")
    if len(inconsistencias) == 0:
        print("   ✅ Lógica de edad y retiro validada. No hay inconsistencias.")
    else:
        print(f"   ❌ ERROR: Se encontraron {len(inconsistencias)} registros incoherentes.")

    # --- PRUEBA 4: Conectividad Total (El JOIN de SQL) ---
    # Esta es la prueba reina: conectar las 3 tablas
    full_warehouse = (df_appointments
                      .merge(df_patients, on='patient_id', how='inner')
                      .merge(df_diagnoses, on='icd10_id', how='inner'))

    print(f"\n4. CONECTIVIDAD TOTAL:")
    print(f"   - Registros recuperados en JOIN triple: {len(full_warehouse):,}")
    print(f"   - Cobertura de la Fact Table: {(len(full_warehouse)/len(df_appointments))*100:.2f}%")
    print("     (Nota: Si es < 100% es por los nulos inyectados para práctica de limpieza)")

if __name__ == "__main__":
    run_audit()

=== AUDITORÍA DE SISTEMA HOSPITALARIO ===

1. ÍNTEGRIDAD REFERENCIAL:
   - IDs de pacientes huérfanos: 0
   - IDs de diagnósticos huérfanos: 0
   ✅ Integridad referencial perfecta.

2. ANÁLISIS DE CARDINALIDAD:
   - Promedio de citas por paciente: 32.00
   - Paciente con más recurrencia: 56 citas
   ✅ Dimensión Pacientes: patient_id es ÚNICO (Relación 1:N confirmada).
   ✅ Dimensión Diagnoses: icd10_id es ÚNICO (Relación 1:N confirmada).

3. COHERENCIA DE NEGOCIO:
   ✅ Lógica de edad y retiro validada. No hay inconsistencias.

4. CONECTIVIDAD TOTAL:
   - Registros recuperados en JOIN triple: 792,000
   - Cobertura de la Fact Table: 99.00%
     (Nota: Si es < 100% es por los nulos inyectados para práctica de limpieza)
